# CIPLab Real and Fake Face Detection Cross-Dataset Benchmark
### Evaluating Final EFFResNet-Unified Architecture on Component Manipulation

**Benchmark Characteristics:**
1. **Dataset**: CIPLab Real and Fake Face Detection (Yonsei University).
2. **Manipulation Type**: Expert digital component morphing (eyes, nose, mouth warping).
3. **Model**: Unified EFFResNet-ViT + 2D-FFT + SRM Steganalysis (`best_effresnet_unified.pth`).
4. **Evaluation**: Zero-shot cross-dataset transfer (no fine-tuning).

In [ ]:
import os, sys, time, json, random
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print('PyTorch Version :', torch.__version__)
print('Device          :', device)
if torch.cuda.is_available():
    print('GPU Device      :', torch.cuda.get_device_name(0))
    print('VRAM Available  :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
print('=' * 65)

In [ ]:
# ── CHECKPOINT DISCOVERY ─────────────────────────────────────────
def find_checkpoint() -> Path:
    for p in Path('/kaggle/input').rglob('best_effresnet_unified.pth'):
        if p.is_file(): return p
    for p in Path('.').rglob('best_effresnet_unified.pth'):
        if p.is_file(): return p
    raise FileNotFoundError('Could not locate best_effresnet_unified.pth under /kaggle/input')

CHECKPOINT_PATH = find_checkpoint()
print('Located Trained Unified Checkpoint at:', CHECKPOINT_PATH)
print(f'File Size: {CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB')

In [ ]:
# ── CIPLAB DATASET DISCOVERY & MANIFEST ──────────────────────────
def locate_ciplab_folders() -> Tuple[Path, Path]:
    # Search for training_real and training_fake
    real_p, fake_p = None, None
    for d in Path('/kaggle/input').rglob('*'):
        if d.is_dir():
            if 'training_real' in d.name.lower() or ('real' in d.name.lower() and 'ciplab' in str(d).lower()):
                real_p = d
            if 'training_fake' in d.name.lower() or ('fake' in d.name.lower() and 'ciplab' in str(d).lower()):
                fake_p = d
    if real_p and fake_p: return real_p, fake_p
    for r in Path('/kaggle/input').rglob('*real*'):
        if r.is_dir():
            parent = r.parent
            for sub in parent.iterdir():
                if sub.is_dir() and 'fake' in sub.name.lower():
                    return r, sub
    raise FileNotFoundError('CIPLab real/fake folders not found under /kaggle/input')

real_dir, fake_dir = locate_ciplab_folders()
print('Discovered Real Folder:', real_dir)
print('Discovered Fake Folder:', fake_dir)

def build_ciplab_manifest(real_dir: Path, fake_dir: Path, seed: int = 42):
    random.seed(seed)
    real_files = sorted([str(p) for p in real_dir.rglob('*.*') if p.suffix.lower() in ['.jpg', '.png', '.jpeg']])
    fake_files = sorted([str(p) for p in fake_dir.rglob('*.*') if p.suffix.lower() in ['.jpg', '.png', '.jpeg']])
    print(f'Total Real Images Discovered: {len(real_files):,}')
    print(f'Total Fake Images Discovered: {len(fake_files):,}')
    entries = [(f, 0) for f in real_files] + [(f, 1) for f in fake_files]
    random.shuffle(entries)
    return entries

test_entries = build_ciplab_manifest(real_dir, fake_dir)
print(f'Total Evaluated CIPLab Benchmark Images: {len(test_entries):,}')

In [ ]:
# ── DATASET & DATALOADER ─────────────────────────────────────────
class CIPLabDataset(Dataset):
    def __init__(self, entries, transform=None):
        self.entries = entries
        self.transform = transform
    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        path, label = self.entries[idx]
        try:    img = Image.open(path).convert('RGB')
        except: img = Image.new('RGB', (224, 224))
        if self.transform: img = self.transform(img)
        else: img = transforms.ToTensor()(img)
        return img, torch.tensor(label, dtype=torch.float32)

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_loader = DataLoader(CIPLabDataset(test_entries, eval_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Evaluation Batches: {len(eval_loader)}')

In [ ]:
# ── MODEL ARCHITECTURE & WEIGHT LOADING ──────────────────────────
def get_srm_filters() -> torch.Tensor:
    f1 = np.array([
        [0,  0,  0,  0,  0],
        [0, -1,  2, -1,  0],
        [0,  2, -4,  2,  0],
        [0, -1,  2, -1,  0],
        [0,  0,  0,  0,  0]
    ], dtype=np.float32) / 4.0
    f2 = np.array([
        [-1,  2, -2,  2, -1],
        [ 2, -6,  8, -6,  2],
        [-2,  8,-12,  8, -2],
        [ 2, -6,  8, -6,  2],
        [-1,  2, -2,  2, -1]
    ], dtype=np.float32) / 12.0
    f3 = np.array([
        [0,  0,  0,  0,  0],
        [0,  1, -2,  1,  0],
        [0, -2,  4, -2,  0],
        [0,  1, -2,  1,  0],
        [0,  0,  0,  0,  0]
    ], dtype=np.float32) / 4.0
    filters = np.stack([f1, f2, f3], axis=0)
    return torch.tensor(np.expand_dims(filters, axis=1), dtype=torch.float32)

class SRMSteganalysisStream(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        srm_k = get_srm_filters()
        self.srm_conv = nn.Conv2d(3, 3, kernel_size=5, padding=2, groups=3, bias=False)
        self.srm_conv.weight.data = srm_k.repeat(1, 1, 1, 1)
        self.srm_conv.weight.requires_grad = False
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, out_dim, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_dim), nn.GELU()
        )
    def forward(self, x): return self.encoder(self.srm_conv(x))

class SpectralFrequencyStream(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, out_dim, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_dim), nn.GELU()
        )
    def forward(self, x):
        fft2d = torch.fft.fft2(x, norm='ortho')
        mag = torch.log(torch.abs(fft2d) + 1e-6)
        mag_shift = torch.fft.fftshift(mag, dim=(-2, -1))
        return self.encoder(mag_shift)

class ConvLocalityBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.GELU()
        self.depthwise = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()
    def forward(self, x): return self.act2(self.bn2(self.depthwise(self.act1(self.bn1(self.conv1(x))))))

class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, kv):
        q_n = self.norm_q(q)
        kv_n = self.norm_kv(kv)
        out, _ = self.mha(q_n, kv_n, kv_n)
        return self.dropout(out)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim), nn.Dropout(dropout)
        )
    def forward(self, x):
        normed = self.norm1(x)
        attn_out, _ = self.mha(normed, normed, normed)
        x = x + self.dropout1(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x

class EFFResNetUnified(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, num_vit_layers=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        eff = models.efficientnet_b0(weights=None)
        self.eff_features = eff.features
        res = models.resnet50(weights=None)
        self.res_stem = nn.Sequential(
            res.conv1, res.bn1, res.relu, res.maxpool,
            res.layer1, res.layer2, res.layer3, res.layer4
        )
        self.fusion_neck = ConvLocalityBlock(1280 + 2048, embed_dim)
        self.freq_stream = SpectralFrequencyStream(embed_dim)
        self.srm_stream = SRMSteganalysisStream(embed_dim)
        self.cross_attn_freq = CrossAttentionBlock(embed_dim, num_heads=num_heads, dropout=dropout)
        self.cross_attn_srm = CrossAttentionBlock(embed_dim, num_heads=num_heads, dropout=dropout)
        self.alpha_freq = nn.Parameter(torch.zeros(1))
        self.alpha_srm = nn.Parameter(torch.zeros(1))
        self.num_patches = 7 * 7
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.transformer_layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_vit_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.post_attn_locality = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, groups=embed_dim, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU()
        )
        self.feat_proj = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 1)
        )
    def forward(self, x):
        B = x.size(0)
        f_eff = self.eff_features(x)
        f_res = self.res_stem(x)
        if f_eff.shape[2:] != f_res.shape[2:]:
            f_eff = F.interpolate(f_eff, size=f_res.shape[2:], mode='bilinear', align_corners=False)
        f_spatial = self.fusion_neck(torch.cat([f_eff, f_res], dim=1))
        f_freq = self.freq_stream(x)
        f_srm = self.srm_stream(x)
        H, W = f_spatial.shape[2], f_spatial.shape[3]
        tokens_spatial = f_spatial.flatten(2).transpose(1, 2)
        tokens_freq = f_freq.flatten(2).transpose(1, 2)
        tokens_srm = f_srm.flatten(2).transpose(1, 2)
        freq_context = self.cross_attn_freq(tokens_spatial, tokens_freq)
        srm_context = self.cross_attn_srm(tokens_spatial, tokens_srm)
        tokens_fused = (
            tokens_spatial
            + torch.tanh(self.alpha_freq) * freq_context
            + torch.tanh(self.alpha_srm) * srm_context
        )
        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens_fused], dim=1)
        tokens = self.pos_dropout(tokens + self.pos_embed[:, :tokens.size(1), :])
        for layer in self.transformer_layers: tokens = layer(tokens)
        tokens = self.norm(tokens)
        cls_out = tokens[:, 0]
        spatial_tokens = tokens[:, 1:]
        spatial_grid = spatial_tokens.transpose(1, 2).reshape(B, self.embed_dim, H, W)
        spatial_recovered = self.post_attn_locality(spatial_grid)
        spatial_pool = F.adaptive_avg_pool2d(spatial_recovered, (1, 1)).flatten(1)
        feat = self.feat_proj(torch.cat([cls_out, spatial_pool], dim=1))
        return self.classifier(feat)

model = EFFResNetUnified(embed_dim=256, num_heads=8, num_vit_layers=4).to(device)
try:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
if 'model_state_dict' in ckpt: model.load_state_dict(ckpt['model_state_dict'])
else: model.load_state_dict(ckpt)
model.eval()
print('Successfully loaded trained EFFResNet-Unified model!')

In [ ]:
# ── CROSS-DATASET INFERENCE EXECUTION ───────────────────────────
all_probs = []
all_labels = []

t0 = time.time()
print('Running inference on unseen CIPLab Real-and-Fake Face dataset...')
with torch.no_grad():
    for step, (imgs, lbls) in enumerate(eval_loader):
        imgs = imgs.to(device, non_blocking=True)
        logits = model(imgs)
        probs = torch.sigmoid(logits)
        probs = torch.nan_to_num(probs, nan=0.5)
        all_probs.extend(probs.squeeze(1).cpu().numpy().tolist())
        all_labels.extend(lbls.numpy().tolist())
        if (step + 1) % 20 == 0 or (step + 1) == len(eval_loader):
            print(f'Processed [{step+1}/{len(eval_loader)}] batches...')

elapsed = time.time() - t0
print(f'Inference finished in {elapsed:.1f}s ({len(all_probs)/elapsed:.1f} frames/sec)')

In [ ]:
# ── METRICS COMPUTATION & BENCHMARK SUMMARY ─────────────────────
y_true = np.array(all_labels)
y_prob = np.nan_to_num(np.array(all_probs), nan=0.5)
y_pred = (y_prob >= 0.5).astype(int)

acc = round(accuracy_score(y_true, y_pred) * 100.0, 2)
prec = round(precision_score(y_true, y_pred, zero_division=0) * 100.0, 2)
rec = round(recall_score(y_true, y_pred, zero_division=0) * 100.0, 2)
f1 = round(f1_score(y_true, y_pred, zero_division=0) * 100.0, 2)
auc = round(roc_auc_score(y_true, y_prob) * 100.0, 2) if len(set(y_true)) > 1 else 50.0
cm = confusion_matrix(y_true, y_pred).tolist()

print('=' * 80)
print('CIPLAB REAL-AND-FAKE FACE DETECTION CROSS-DATASET RESULTS')
print('=' * 80)
print(f'Accuracy   : {acc}%')
print(f'Precision  : {prec}%')
print(f'Recall     : {rec}% (Manipulated Face Catch Rate)')
print(f'F1-Score   : {f1}%')
print(f'AUC-ROC    : {auc}%')
print(f'Confusion Matrix: {cm}')
print('=' * 80)

OUTPUT_DIR = Path('/kaggle/working/output_ciplab_eval')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1_score': f1,
    'auc_roc': auc,
    'confusion_matrix': cm,
    'total_evaluated_images': len(y_true)
}
with open(OUTPUT_DIR / 'ciplab_eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Confusion Matrix plot
plt.figure(figsize=(6, 5))
sns.heatmap(np.array(cm), annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title(f'CIPLab Real/Fake Face Detection (Acc: {acc}%, AUC: {auc}%)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ciplab_confusion_matrix.png', dpi=300)
plt.show()
print('Saved results and plots to', OUTPUT_DIR)